In [7]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split



In [8]:

df = pd.read_csv(r"iris.csv")

X_raw = df.iloc[:, :-1].values
y_raw = df.iloc[:, -1].values

le = LabelEncoder()
y = le.fit_transform(y_raw)

a, b = 0, 10
scaler = MinMaxScaler(feature_range=(a, b))
X = scaler.fit_transform(X_raw)

X_new_raw = np.array([[6, 3, 0.7, 0.9]])
X_new = scaler.transform(X_new_raw)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)



A. Naive Bayes

In [9]:
# Chay

def gaussian(x, mean, var):
    eps = 1e-9
    coeff = 1.0 / np.sqrt(2.0 * np.pi * ( var + eps))
    exponent = np.exp(- (x - mean) ** 2 / (2 * ( var + eps )))
    return coeff * exponent

classes = np.unique(y)
probs = {}
for c in classes:
    X_c = X[y == c]
    prior = len(X_c) / len(X)
    mean = X_c.mean(axis = 0)
    var = X_c.var(axis = 0)

    likelihood = np.prod(gaussian(X_new[0], mean, var))
    probs[c] = prior * likelihood

total = sum(probs.values())
for c in classes:
    probs[c] /= total
    print(f"Xac xuat thuoc lop {le.inverse_transform([c])[0]}: {probs[c]: .4f}")

pred_class = max( probs, key = probs.get)
print("X thuoc lop: ", le.inverse_transform([pred_class])[0])



Xac xuat thuoc lop Iris-setosa:  0.5157
Xac xuat thuoc lop Iris-versicolor:  0.4843
Xac xuat thuoc lop Iris-virginica:  0.0000
X thuoc lop:  Iris-setosa


In [10]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()
nb.fit(X,y)
print("Xac suat (thu vien):", nb.predict_proba(X_new))
print("X thuoc lop:", le.inverse_transform(nb.predict(X_new))[0])

Xac suat (thu vien): [[5.15731924e-01 4.84268060e-01 1.57371259e-08]]
X thuoc lop: Iris-setosa


B. Phân cụm K-Means



In [10]:
K = 2
np.random.seed(42)

centroids = X[np.random.choice(len(X), K , replace = False)]

for _ in range(100):
    distance = np.linalg.norm(X[:, np.newaxis] - centroids, axis = 2)
    labels = np.argmin( distance, axis = 1)

    new_centroids = np.array([X[labels == k].mean(axis = 0) for k in range(K)])
    if np.allclose(centroids, new_centroids):
        break
    centroids = new_centroids

for k in range(K):
    print(f"Cum {k}: {np.sum(labels == k)} mau | Tam cum: {np.round(centroids[k], 3)}")

dist_new = np.linalg.norm(centroids - X_new[0], axis = 1)
print("X_new thuoc cum: ", np.argmin(dist_new))

Cum 0: 100 mau | Tam cum: [5.45  3.633 6.62  6.567]
Cum 1: 50 mau | Tam cum: [1.961 5.908 0.786 0.6  ]
X_new thuoc cum:  1


In [13]:
from sklearn.cluster import KMeans

km = KMeans( n_clusters = K, random_state = 42, n_init = 10 )
labels = km.fit_predict(X)

for k in range(K):
    print(f"Cum {k}: {np.sum(labels == k)} mau | Tam cum: {np.round(centroids[k], 3)}")

print("X_new thuoc cum: ", km.predict(X_new)[0])

Cum 0: 100 mau | Tam cum: [5.45  3.633 6.62  6.567]
Cum 1: 50 mau | Tam cum: [1.961 5.908 0.786 0.6  ]
X_new thuoc cum:  1


C. Phân cụm phân cấp - Thuât toán gộp

In [15]:
K = 3

clusters = [[i] for i in range(len(X))]
centers = [X[i] for i in range(len(X))]

while len(clusters) > K:
    min_dist = float('inf')
    pair = (0, 1)
    for i in range(len(centers)):
        for j in range( i + 1, len(centers)):
            d = np.linalg.norm(centers[i] - centers[j])
            if d < min_dist:
                min_dist = d
                pair = (i, j)

    i, j = pair
    clusters[i].extend(clusters[j])
    centers[i] = X[clusters[i]].mean(axis = 0)
    clusters.pop(j)
    centers.pop(j)

for idx, c in enumerate(clusters):
    print(f"Cum {idx}: {len(c)} mau")

dist_new = [np.linalg.norm(c - X_new[0]) for c in centers]
print("X_new thuoc cum:", np.argmin(dist_new))

Cum 0: 49 mau
Cum 1: 1 mau
Cum 2: 100 mau
X_new thuoc cum: 0


In [17]:
from sklearn.cluster import AgglomerativeClustering

agg = AgglomerativeClustering(n_clusters = K, linkage='ward')
labels = agg.fit_predict(X)

centers = []
for k in range(K):
    print(f"Cum {k}: {np.sum(labels == k )} mau")
    centers.append(X[labels == k].mean(axis = 0))

dist_new = [np.linalg.norm(c - X_new[0]) for c in centers]
print("X thuoc cum: ", np.argmin(dist_new))


Cum 0: 67 mau
Cum 1: 50 mau
Cum 2: 33 mau
X thuoc cum:  1


D. Cây quyết định

In [18]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

clf = DecisionTreeClassifier(criterion = 'entropy', random_state = 42 )

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Du doan X_new: ", le.inverse_transform(clf.predict(X_new))[0])

Accuracy:  1.0
Du doan X_new:  Iris-setosa


E. Mạng nơ-ron

In [22]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(hidden_layer_sizes = (10, 10), activation='relu', max_iter = 500, random_state=42)
mlp.fit(X_train, y_train)
print("Accuracy", mlp.score(X_test, y_test))
print("Du doan X_new:", le.inverse_transform(mlp.predict(X_new))[0])

Accuracy 0.9666666666666667
Du doan X_new: Iris-setosa


C:\Users\admin\Desktop\MonTrenLop\KPDL\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [24]:
w = np.zeros(X_train.shape[1])
b= 0.0
lr = 0.01

for _ in range(100):
    for xi, yi in zip(X_train, y_train):
        y_hat = 1 if (np.dot(xi, w) + b ) >= 0 else 0
        w += lr*(yi - y_hat)*xi
        b += lr*(yi - y_hat)

pred_new = 1 if ( np.dot(X_new[0], w) + b ) >= 0 else 0
print("Du doan X_new: ", pred_new)

Du doan X_new:  1


E. Luat ket hop

In [27]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

transaction = [[str(item) for item in row if pd.notna(item)] for row in df.values]

te = TransactionEncoder()
df_encoder = pd.DataFrame(te.fit_transform(transaction), columns = te.columns_)

freq_items = fpgrowth(df_encoder, min_support=0.25, use_colnames=True)

rules = association_rules(freq_items, metric="lift", min_threshold=1.0)
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head())

Empty DataFrame
Columns: [antecedents, consequents, support, confidence, lift]
Index: []
